# NB15: Pipeline-Derived Functional Connectivity

> **Local pipeline required.** This notebook requires fMRIPrep BOLD outputs and cannot run in Google Colab. See NB00--NB06 and NB08 for Colab-compatible notebooks.

## Purpose

Compute functional connectivity (FC) matrices directly from the
preprocessing pipeline outputs (fMRIPrep BOLD + FreeSurfer parcellation),
rather than relying on pre-computed TVB derivatives.

### Relationship to NB04

This notebook is the **Track B** (full pipeline) counterpart to
**NB04** (Track A, educational). NB04 teaches BOLD/HRF concepts using
pre-computed TVB/rsHRF derivatives that work in Colab. This notebook
computes the same FC matrices from scratch using the pipeline's
fMRIPrep outputs, giving you full control over confound regression
and parcellation choices.

### What You Need

- fMRIPrep outputs: `derivatives/fmriprep/sub-{SUB}/ses-{SES}/func/`
- Registered parcellation: `derivatives/registered_t1w/sub-{SUB}/ses-{SES}/aparc+aseg_space-T1w.nii.gz`
- Run the pipeline first: `bash scripts/run_pipeline.sh --clinical-postop --parallel`

### FC Computation Steps

1. Load fMRIPrep preprocessed BOLD (T1w space)
2. Load confound regressors and select nuisance variables
3. Load aparc+aseg parcellation (registered to T1w space)
4. Extract mean ROI time series (Desikan-Killiany atlas, 68 cortical + subcortical)
5. Regress out confounds (motion, aCompCor, drift)
6. Bandpass filter (0.01-0.1 Hz)
7. Compute Pearson correlation FC matrix
8. Compare with NB04's TVB-derived FC

In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import json

if 'google.colab' in sys.modules:
    print('NOTE: This notebook requires local pipeline outputs (fMRIPrep, FreeSurfer).')
    print('      See NB04 for a Colab-compatible version using pre-computed TVB derivatives.')

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / 'pyproject.toml').exists() else _cwd.parent

DERIV_ROOT = REPO_ROOT / 'derivatives'
FMRIPREP_DIR = DERIV_ROOT / 'fmriprep'
REG_DIR = DERIV_ROOT / 'registered_t1w'
FC_OUT = DERIV_ROOT / 'connectivity'

SUBJECTS = ['PAT16']
SESSIONS = ['preop', 'postop']

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f'Derivatives: {DERIV_ROOT}')

Derivatives: <REPO_ROOT>/derivatives


## 1. Load fMRIPrep BOLD and Confounds

In [2]:
try:
    import nibabel as nib
    import pandas as pd
    from scipy import signal
except ImportError as e:
    print(f'Missing dependency: {e}. Install with: pip install nibabel pandas scipy')
    raise

def load_bold_and_confounds(sub, ses):
    """Load fMRIPrep BOLD (T1w space) and confound regressors."""
    sub_ses = f'sub-{sub}_ses-{ses}'
    func_dir = FMRIPREP_DIR / f'sub-{sub}' / f'ses-{ses}' / 'func'

    bold_files = list(func_dir.glob('*_space-T1w_desc-preproc_bold.nii.gz'))
    conf_files = list(func_dir.glob('*_desc-confounds_timeseries.tsv'))
    json_files = list(func_dir.glob('*_space-T1w_desc-preproc_bold.json'))

    if not bold_files:
        return None, None, None

    bold_img = nib.load(str(bold_files[0]))
    confounds = pd.read_csv(str(conf_files[0]), sep='\t') if conf_files else None

    tr = None
    if json_files:
        with open(json_files[0]) as f:
            meta = json.load(f)
            tr = meta.get('RepetitionTime')

    return bold_img, confounds, tr

# Check availability
for sub in SUBJECTS:
    for ses in SESSIONS:
        bold_img, confounds, tr = load_bold_and_confounds(sub, ses)
        if bold_img is not None:
            print(f'sub-{sub} ses-{ses}: BOLD shape={bold_img.shape}, '
                  f'TR={tr}s, confounds={len(confounds.columns)} cols')
        else:
            print(f'sub-{sub} ses-{ses}: not available')

sub-PAT16 ses-preop: BOLD shape=(51, 62, 50, 180), TR=2.4s, confounds=213 cols
sub-PAT16 ses-postop: BOLD shape=(51, 62, 51, 180), TR=2.4s, confounds=213 cols


## 2. Extract ROI Time Series from aparc+aseg

The Desikan-Killiany atlas (aparc+aseg) provides 68 cortical ROIs +
subcortical structures. We extract the mean BOLD time series for each ROI.

In [3]:
DK_CORTICAL = {
    1001: 'L-bankssts', 1002: 'L-caudalanteriorcingulate',
    1003: 'L-caudalmiddlefrontal', 1005: 'L-cuneus',
    1006: 'L-entorhinal', 1007: 'L-fusiform',
    1008: 'L-inferiorparietal', 1009: 'L-inferiortemporal',
    1010: 'L-isthmuscingulate', 1011: 'L-lateraloccipital',
    1012: 'L-lateralorbitofrontal', 1013: 'L-lingual',
    1014: 'L-medialorbitofrontal', 1015: 'L-middletemporal',
    1016: 'L-parahippocampal', 1017: 'L-paracentral',
    1018: 'L-parsopercularis', 1019: 'L-parsorbitalis',
    1020: 'L-parstriangularis', 1021: 'L-pericalcarine',
    1022: 'L-postcentral', 1023: 'L-posteriorcingulate',
    1024: 'L-precentral', 1025: 'L-precuneus',
    1026: 'L-rostralanteriorcingulate', 1027: 'L-rostralmiddlefrontal',
    1028: 'L-superiorfrontal', 1029: 'L-superiorparietal',
    1030: 'L-superiortemporal', 1031: 'L-supramarginal',
    1032: 'L-frontalpole', 1033: 'L-temporalpole',
    1034: 'L-transversetemporal', 1035: 'L-insula',
}
# Right hemisphere: same labels + 1000
DK_CORTICAL.update({k + 1000: name.replace('L-', 'R-')
                    for k, name in list(DK_CORTICAL.items())})

DK_SUBCORTICAL = {
    10: 'L-Thalamus', 11: 'L-Caudate', 12: 'L-Putamen',
    13: 'L-Pallidum', 17: 'L-Hippocampus', 18: 'L-Amygdala',
    26: 'L-Accumbens',
    49: 'R-Thalamus', 50: 'R-Caudate', 51: 'R-Putamen',
    52: 'R-Pallidum', 53: 'R-Hippocampus', 54: 'R-Amygdala',
    58: 'R-Accumbens',
}

ALL_ROIS = {**DK_CORTICAL, **DK_SUBCORTICAL}
print(f'Total ROIs: {len(ALL_ROIS)} ({len(DK_CORTICAL)} cortical + {len(DK_SUBCORTICAL)} subcortical)')

Total ROIs: 82 (68 cortical + 14 subcortical)


In [4]:
def extract_roi_timeseries(bold_img, aparc_img, roi_labels, min_voxels=10):
    """Extract mean BOLD time series per ROI from aparc+aseg parcellation."""
    bold_data = np.asarray(bold_img.dataobj)
    aparc_data = np.asarray(aparc_img.dataobj).astype(int)

    n_vols = bold_data.shape[3]
    roi_ids = sorted(roi_labels.keys())
    roi_names = [roi_labels[r] for r in roi_ids]
    ts = np.zeros((n_vols, len(roi_ids)))
    valid = []

    for i, roi_id in enumerate(roi_ids):
        mask = aparc_data == roi_id
        n_vox = mask.sum()
        if n_vox >= min_voxels:
            ts[:, i] = bold_data[mask].mean(axis=0)
            valid.append(i)
        else:
            ts[:, i] = np.nan

    return ts, roi_names, valid

# Test on first available subject
for sub in SUBJECTS:
    for ses in SESSIONS:
        bold_img, confounds, tr = load_bold_and_confounds(sub, ses)
        aparc_path = REG_DIR / f'sub-{sub}' / f'ses-{ses}' / 'aparc+aseg_space-T1w.nii.gz'
        if bold_img is not None and aparc_path.exists():
            aparc_img = nib.load(str(aparc_path))
            if bold_img.shape[:3] == aparc_img.shape[:3]:
                ts, names, valid = extract_roi_timeseries(bold_img, aparc_img, ALL_ROIS)
                print(f'sub-{sub} ses-{ses}: {len(valid)}/{len(ALL_ROIS)} ROIs with data, '
                      f'{ts.shape[0]} timepoints')
            else:
                print(f'sub-{sub} ses-{ses}: shape mismatch '
                      f'BOLD={bold_img.shape[:3]} vs aparc={aparc_img.shape}')
        else:
            print(f'sub-{sub} ses-{ses}: not available')
        break
    break

sub-PAT16 ses-preop: shape mismatch BOLD=(51, 62, 50) vs aparc=(256, 256, 256)


## 3. Confound Regression and Bandpass Filtering

Standard resting-state FC preprocessing:
1. Regress out motion parameters (6 DOF + derivatives)
2. Regress out aCompCor components (WM/CSF signals)
3. Bandpass filter to 0.01-0.1 Hz
4. Scrub high-motion volumes (FD > 0.5 mm)

In [5]:
CONFOUND_COLS = [
    'trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z',
    'trans_x_derivative1', 'trans_y_derivative1', 'trans_z_derivative1',
    'rot_x_derivative1', 'rot_y_derivative1', 'rot_z_derivative1',
    'a_comp_cor_00', 'a_comp_cor_01', 'a_comp_cor_02',
    'a_comp_cor_03', 'a_comp_cor_04',
    'cosine00', 'cosine01', 'cosine02',
]

def clean_timeseries(ts, confounds, tr, valid_rois,
                     bandpass=(0.01, 0.1), fd_thresh=0.5):
    """Regress confounds, bandpass filter, and scrub high-motion volumes."""
    available_cols = [c for c in CONFOUND_COLS if c in confounds.columns]
    X = confounds[available_cols].fillna(0).values

    ts_clean = ts.copy()
    for i in valid_rois:
        y = ts[:, i]
        if np.isnan(y).any():
            continue
        # Regress confounds
        X_design = np.column_stack([X, np.ones(len(y))])
        beta = np.linalg.lstsq(X_design, y, rcond=None)[0]
        residual = y - X_design @ beta

        # Bandpass filter
        if tr and bandpass:
            nyq = 0.5 / tr
            low, high = bandpass[0] / nyq, min(bandpass[1] / nyq, 0.99)
            if low < high:
                b, a = signal.butter(3, [low, high], btype='band')
                residual = signal.filtfilt(b, a, residual)

        ts_clean[:, i] = residual

    # Scrub high-motion volumes
    if 'framewise_displacement' in confounds.columns:
        fd = confounds['framewise_displacement'].fillna(0).values
        bad = fd > fd_thresh
        n_scrubbed = bad.sum()
        ts_clean[bad, :] = np.nan
    else:
        n_scrubbed = 0

    return ts_clean, n_scrubbed

print(f'Confound columns used: {len(CONFOUND_COLS)}')
print(f'Bandpass: 0.01-0.1 Hz, FD threshold: 0.5 mm')

Confound columns used: 20
Bandpass: 0.01-0.1 Hz, FD threshold: 0.5 mm


## 4. Compute FC Matrix

FC is the Pearson correlation between cleaned ROI time series.
We apply Fisher z-transform for subsequent statistical comparisons.

In [6]:
def compute_fc(ts_clean, valid_rois):
    """Compute Pearson correlation FC matrix from cleaned time series."""
    # Remove NaN timepoints (scrubbed volumes)
    good_tps = ~np.any(np.isnan(ts_clean[:, valid_rois]), axis=1)
    ts_good = ts_clean[good_tps][:, valid_rois]

    if ts_good.shape[0] < 30:
        print(f'WARNING: only {ts_good.shape[0]} clean volumes remaining')
        return None

    fc = np.corrcoef(ts_good.T)
    np.fill_diagonal(fc, 0)

    # Fisher z-transform
    fc_z = np.arctanh(np.clip(fc, -0.999, 0.999))

    return fc, fc_z

# Compute and display FC for all available sessions
fc_results = {}
for sub in SUBJECTS:
    for ses in SESSIONS:
        bold_img, confounds, tr = load_bold_and_confounds(sub, ses)
        aparc_path = REG_DIR / f'sub-{sub}' / f'ses-{ses}' / 'aparc+aseg_space-T1w.nii.gz'

        if bold_img is None or not aparc_path.exists():
            print(f'sub-{sub} ses-{ses}: skipped (data not available)')
            continue

        aparc_img = nib.load(str(aparc_path))
        if bold_img.shape[:3] != aparc_img.shape[:3]:
            print(f'sub-{sub} ses-{ses}: skipped (shape mismatch)')
            continue

        ts, names, valid = extract_roi_timeseries(bold_img, aparc_img, ALL_ROIS)
        ts_clean, n_scrub = clean_timeseries(ts, confounds, tr, valid)
        result = compute_fc(ts_clean, valid)

        if result is not None:
            fc, fc_z = result
            fc_results[(sub, ses)] = {'fc': fc, 'fc_z': fc_z,
                                      'names': [names[i] for i in valid],
                                      'n_scrub': n_scrub}
            print(f'sub-{sub} ses-{ses}: FC matrix {fc.shape}, '
                  f'{n_scrub} volumes scrubbed')

print(f'\nFC matrices computed: {len(fc_results)}')

sub-PAT16 ses-preop: skipped (shape mismatch)
sub-PAT16 ses-postop: skipped (shape mismatch)

FC matrices computed: 0


## 5. Visualize FC Matrices

In [7]:
n_plots = len(fc_results)
if n_plots > 0:
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 6))
    if n_plots == 1:
        axes = [axes]

    for idx, ((sub, ses), data) in enumerate(sorted(fc_results.items())):
        ax = axes[idx]
        im = ax.imshow(data['fc'], cmap='RdBu_r', vmin=-0.8, vmax=0.8)
        ax.set_title(f'sub-{sub} ses-{ses}\n({len(data["names"])} ROIs)')
        ax.set_xlabel('ROI')
        ax.set_ylabel('ROI')
        plt.colorbar(im, ax=ax, fraction=0.046, label='Pearson r')

    plt.suptitle('Pipeline-Derived Functional Connectivity', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No FC matrices available. Run the preprocessing pipeline first.')

No FC matrices available. Run the preprocessing pipeline first.


## 6. Save FC Matrices

Save for use in NB16 (SC-FC coupling) and NB17 (longitudinal comparison).

In [8]:
for (sub, ses), data in fc_results.items():
    out_dir = FC_OUT / f'sub-{sub}' / f'ses-{ses}'
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'fc_pearson.npy', data['fc'])
    np.save(out_dir / 'fc_fisher_z.npy', data['fc_z'])
    with open(out_dir / 'fc_roi_names.json', 'w') as f:
        json.dump(data['names'], f, indent=2)
    print(f'Saved: {out_dir}')

if not fc_results:
    print('Nothing to save.')

Nothing to save.
